# Imports

In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load Processed Sentiment, Price Data, and Membership Panel

In [12]:
df_sent = pd.read_excel(r"..\..\Data\Main\Sentiment_Final.xlsx")
prices = pd.read_csv(r"..\..\Data\Main\sp500_prices_2020_2025.csv")
panel = pd.read_csv(r"..\..\Data\Main\membership_panel_clean.csv")

print(f"Sentiment: {df_sent.shape} ({len([c for c in df_sent.columns if c.endswith('_sentiment')])} tickers)")
print(f"Prices: {prices.shape}, {prices['ticker'].nunique()} tickers")
print(f"Panel: {panel.shape}")

Sentiment: (1564, 1122) (561 tickers)
Prices: (722756, 7), 560 tickers
Panel: (606, 5)


## Check Ticker Overlap Between Sentiment and Prices

In [13]:
sent_tickers = set([c.replace('_sentiment', '') for c in df_sent.columns if c.endswith('_sentiment')])
price_tickers = set(prices['ticker'].unique())

in_sent_not_prices = sent_tickers - price_tickers
in_prices_not_sent = price_tickers - sent_tickers

print(f"Sentiment tickers: {len(sent_tickers)}")
print(f"Price tickers: {len(price_tickers)}")
print(f"In both: {len(sent_tickers & price_tickers)}")

print(f"\nIn sentiment but NOT in prices ({len(in_sent_not_prices)}):")
for t in sorted(in_sent_not_prices):
    print(f"  {t}")

print(f"\nIn prices but NOT in sentiment ({len(in_prices_not_sent)}):")
for t in sorted(in_prices_not_sent):
    print(f"  {t}")

Sentiment tickers: 561
Price tickers: 560
In both: 560

In sentiment but NOT in prices (1):
  SBNY

In prices but NOT in sentiment (0):


### Overlap Findings

SBNY (Signature Bank) has Bloomberg sentiment data but no Yahoo Finance price data — the bank collapsed in March 2023 and Yahoo Finance removed its historical prices. Already documented and investigated in 02_Download_Prices. SBNY will be excluded naturally when we inner join sentiment and prices.

All 560 price tickers have matching sentiment data. The merge will proceed with 560 tickers maximum.

# Reshape Sentiment from Wide to Long Format

In [14]:
sent_cols = [c for c in df_sent.columns if c.endswith('_sentiment')]
tickers = [c.replace('_sentiment', '') for c in sent_cols]

all_long = []
for t in tickers:
    date_col = f"{t}_date"
    sent_col = f"{t}_sentiment"
    
    sub = df_sent[[date_col, sent_col]].dropna(subset=[date_col]).copy()
    sub.columns = ['date', 'sentiment']
    sub['ticker'] = t
    all_long.append(sub)

sentiment_long = pd.concat(all_long, ignore_index=True)
sentiment_long['date'] = pd.to_datetime(sentiment_long['date'])

print(f"Shape: {sentiment_long.shape}")
print(f"Tickers: {sentiment_long['ticker'].nunique()}")
print(f"Date range: {sentiment_long['date'].min()} to {sentiment_long['date'].max()}")
print(sentiment_long.head())

Shape: (609750, 3)
Tickers: 547
Date range: 2020-01-02 00:00:00 to 2025-12-31 00:00:00
        date  sentiment ticker
0 2020-01-02     0.0000      A
1 2020-01-03     0.0000      A
2 2020-01-07     0.3292      A
3 2020-01-08     0.0000      A
4 2020-01-09    -0.6269      A


# Merge Sentiment with Prices

In [15]:
prices['date'] = pd.to_datetime(prices['date'])

merged = pd.merge(sentiment_long, prices, on=['date', 'ticker'], how='inner')

print(f"Shape: {merged.shape}")
print(f"Tickers: {merged['ticker'].nunique()}")
print(f"Date range: {merged['date'].min()} to {merged['date'].max()}")
print(f"Columns: {list(merged.columns)}")

Shape: (593458, 8)
Tickers: 546
Date range: 2020-01-02 00:00:00 to 2025-12-31 00:00:00
Columns: ['date', 'sentiment', 'ticker', 'Close', 'High', 'Low', 'Open', 'Volume']


In [16]:
sent_tickers = set(sentiment_long['ticker'].unique())
merged_tickers = set(merged['ticker'].unique())

dropped = sent_tickers - merged_tickers
print(f"Ticker in sentiment but not in merged: {dropped}")

for t in dropped:
    print(f"\n{t}:")
    print(f"  Sentiment dates: {sentiment_long[sentiment_long['ticker']==t]['date'].min()} to {sentiment_long[sentiment_long['ticker']==t]['date'].max()}")
    p = panel[panel['yf_ticker']==t]
    if len(p) > 0:
        print(f"  Panel: first_seen={p.iloc[0]['first_seen']}, last_seen={p.iloc[0]['last_seen']}, quarters={p.iloc[0]['quarters_present']}")
    print(f"  In price data: {t in set(prices['ticker'].unique())}")

Ticker in sentiment but not in merged: {'SBNY'}

SBNY:
  Sentiment dates: 2022-01-05 00:00:00 to 2022-12-27 00:00:00
  Panel: first_seen=2022-01-03, last_seen=2023-01-03, quarters=5
  In price data: False


## Apply Membership Filter

In [17]:
print("BF-B in panel:", 'BF-B' in panel['yf_ticker'].values)
print("BRK-B in panel:", 'BRK-B' in panel['yf_ticker'].values)
print("BF-B in prices:", 'BF-B' in prices['ticker'].values)
print("BRK-B in prices:", 'BRK-B' in prices['ticker'].values)
print("BF-B in merged:", 'BF-B' in merged['ticker'].values)
print("BRK-B in merged:", 'BRK-B' in merged['ticker'].values)

BF-B in panel: True
BRK-B in panel: True
BF-B in prices: True
BRK-B in prices: True
BF-B in merged: True
BRK-B in merged: True


In [18]:
panel['first_seen'] = pd.to_datetime(panel['first_seen'])
panel['last_seen'] = pd.to_datetime(panel['last_seen'])

merged = merged.merge(panel[['yf_ticker', 'first_seen', 'last_seen']], 
                       left_on='ticker', right_on='yf_ticker', how='left')

before = len(merged)
merged = merged[(merged['date'] >= merged['first_seen']) & (merged['date'] <= merged['last_seen'])]
after = len(merged)

print(f"Before membership filter: {before}")
print(f"After membership filter: {after}")
print(f"Rows removed: {before - after}")
print(f"Tickers: {merged['ticker'].nunique()}")

# Drop the extra columns
merged = merged.drop(columns=['yf_ticker', 'first_seen', 'last_seen'])

Before membership filter: 593458
After membership filter: 593458
Rows removed: 0
Tickers: 546


## Apply Minimum Sentiment Coverage Threshold

Remove tickers with fewer than 100 non-zero sentiment days — insufficient coverage for meaningful analysis.

In [19]:
import pandas as pd
import numpy as np

# Load the merged data BEFORE the 100-day threshold was applied
# You need the 546-ticker version, not the 523-ticker version
# If you don't have it saved separately, rebuild it:
# merged = pd.read_csv(r"..\..\Data\Main\merged_data.csv")  # adjust path if needed

# If merged_data.csv is already the 523-ticker version, you'll need to load
# from the step before filtering. Check:
print(f"Tickers in file: {merged['ticker'].nunique()}")
print("If this says 523, you need the pre-filtered version.")
print("If this says 546, you're good.\n")

# --- If you only have the 523 version, reload and redo the merge ---
# Uncomment the block below if needed:
#
# sentiment_long = pd.read_csv("path_to_sentiment_long.csv")  
# prices = pd.read_csv(r"..\..\Data\Main\sp500_prices_2020_2025.csv")
# prices['date'] = pd.to_datetime(prices['date'])
# sentiment_long['date'] = pd.to_datetime(sentiment_long['date'])
# merged = pd.merge(sentiment_long, prices, on=['date', 'ticker'], how='inner')

# === ANALYSIS: Compare 100-day threshold vs coverage-based threshold ===

# For each ticker: count total days, non-zero days, and coverage percentage
ticker_stats = merged.groupby('ticker').agg(
    total_days=('sentiment', 'count'),
    non_zero_days=('sentiment', lambda x: (x != 0).sum())
).reset_index()
ticker_stats['coverage_pct'] = (ticker_stats['non_zero_days'] / ticker_stats['total_days'] * 100).round(2)

print("=== TICKER COVERAGE STATS ===")
print(f"Total tickers: {len(ticker_stats)}")
print(f"\nCoverage distribution:")
print(ticker_stats['coverage_pct'].describe().round(2))

# Method 1: Fixed 100-day threshold
removed_100day = set(ticker_stats[ticker_stats['non_zero_days'] < 100]['ticker'])
print(f"\n=== METHOD 1: 100 non-zero days threshold ===")
print(f"Tickers removed: {len(removed_100day)}")

# Method 2: Test multiple percentage thresholds
print(f"\n=== METHOD 2: Coverage-based thresholds ===")
for pct in [5, 10, 15, 20, 25, 30]:
    removed_pct = set(ticker_stats[ticker_stats['coverage_pct'] < pct]['ticker'])
    overlap = removed_pct & removed_100day
    only_in_pct = removed_pct - removed_100day
    only_in_100 = removed_100day - removed_pct
    print(f"\n  {pct}% threshold:")
    print(f"    Removed: {len(removed_pct)} tickers")
    print(f"    Same as 100-day: {len(overlap)}")
    print(f"    Extra (not in 100-day): {len(only_in_pct)} {sorted(only_in_pct) if only_in_pct else ''}")
    print(f"    Missed (in 100-day but kept): {len(only_in_100)} {sorted(only_in_100) if only_in_100 else ''}")
    if removed_pct == removed_100day:
        print(f"    >>> EXACT MATCH with 100-day threshold <<<")

# Show the 23 tickers removed by 100-day with their coverage stats
print(f"\n=== DETAIL: Tickers removed by 100-day threshold ===")
removed_detail = ticker_stats[ticker_stats['non_zero_days'] < 100].sort_values('non_zero_days')
print(removed_detail[['ticker', 'total_days', 'non_zero_days', 'coverage_pct']].to_string(index=False))

Tickers in file: 546
If this says 523, you need the pre-filtered version.
If this says 546, you're good.

=== TICKER COVERAGE STATS ===
Total tickers: 546

Coverage distribution:
count    546.00
mean      72.95
std       12.71
min       19.83
25%       70.21
50%       75.89
75%       79.90
max       99.76
Name: coverage_pct, dtype: float64

=== METHOD 1: 100 non-zero days threshold ===
Tickers removed: 23

=== METHOD 2: Coverage-based thresholds ===

  5% threshold:
    Removed: 0 tickers
    Same as 100-day: 0
    Extra (not in 100-day): 0 
    Missed (in 100-day but kept): 23 ['AIV', 'APO', 'BFH', 'BIO', 'COTY', 'CPRI', 'EXE', 'FLS', 'HOG', 'HP', 'HRB', 'KSS', 'LEG', 'LII', 'M', 'NOV', 'OGN', 'PRGO', 'SLG', 'TKO', 'UNM', 'WSM', 'XRX']

  10% threshold:
    Removed: 0 tickers
    Same as 100-day: 0
    Extra (not in 100-day): 0 
    Missed (in 100-day but kept): 23 ['AIV', 'APO', 'BFH', 'BIO', 'COTY', 'CPRI', 'EXE', 'FLS', 'HOG', 'HP', 'HRB', 'KSS', 'LEG', 'LII', 'M', 'NOV', 'OGN', 'P

In [9]:
non_zero_counts = merged[merged['sentiment'] != 0].groupby('ticker').size()
low_coverage = non_zero_counts[non_zero_counts < 100].index.tolist()

print(f"Tickers below 100 non-zero sentiment days: {len(low_coverage)}")
for t in low_coverage:
    print(f"  {t}: {non_zero_counts[t]} non-zero days")

before = merged['ticker'].nunique()
merged = merged[~merged['ticker'].isin(low_coverage)]
after = merged['ticker'].nunique()

print(f"\nTickers before: {before}")
print(f"Tickers after: {after}")
print(f"Tickers removed: {before - after}")
print(f"Final shape: {merged.shape}")

Tickers below 100 non-zero sentiment days: 23
  AIV: 25 non-zero days
  APO: 97 non-zero days
  BFH: 14 non-zero days
  BIO: 91 non-zero days
  COTY: 62 non-zero days
  CPRI: 37 non-zero days
  EXE: 44 non-zero days
  FLS: 40 non-zero days
  HOG: 32 non-zero days
  HP: 23 non-zero days
  HRB: 35 non-zero days
  KSS: 69 non-zero days
  LEG: 75 non-zero days
  LII: 96 non-zero days
  M: 52 non-zero days
  NOV: 99 non-zero days
  OGN: 87 non-zero days
  PRGO: 93 non-zero days
  SLG: 86 non-zero days
  TKO: 47 non-zero days
  UNM: 77 non-zero days
  WSM: 45 non-zero days
  XRX: 80 non-zero days

Tickers before: 546
Tickers after: 523
Tickers removed: 23
Final shape: (589886, 8)


# Summary and Save

In [10]:
non_zero = (merged['sentiment'] != 0).sum()
total = len(merged)

print(f"=== Final Merged Dataset ===")
print(f"Shape: {merged.shape}")
print(f"Tickers: {merged['ticker'].nunique()}")
print(f"Date range: {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"Non-zero sentiment: {non_zero:,} ({non_zero/total*100:.1f}%)")
print(f"Zero sentiment: {total - non_zero:,} ({(total-non_zero)/total*100:.1f}%)")

merged.to_csv(r"..\..\Data\Main\merged_data.csv", index=False)
print(f"\nSaved: merged_data.csv")

=== Final Merged Dataset ===
Shape: (589886, 8)
Tickers: 523
Date range: 2020-01-02 to 2025-12-31
Non-zero sentiment: 445,229 (75.5%)
Zero sentiment: 144,657 (24.5%)

Saved: merged_data.csv
